# 解释那些只被一种富集方法或者一种酶切方法鉴定到的小肽

In [1]:
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")
source("~/bin/lit_utils.R")
lib_text()
lib_plot()
setwd("/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/")
output_path <- "./stat_output/S7"
create_path(output_path)
fread_c("./stat_output/S1/sample_metadata_ordered.txt") -> sample_metadata
# 去掉PCP第4个重复
sample_metadata %>% filter(Replicate!=4)  %>% filter(Eyzyme!='Null') -> sample_metadata
psm_sep_all_path <- '/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/stat_output/S6/psm_sep_all.txt'
fread_c(psm_sep_all_path) -> psm_sep_all
psm_sep_all %>% filter(PSM_type=="Unique") -> psm_sep_unique
# 去掉PCP第4个重复
psm_sep_unique %>% filter(!grepl("4_PCP",psm_sep_unique$Sample) & !grepl("less3K",psm_sep_unique$Sample)) -> psm_sep_unique
psm_sep_unique %>% distinct(Protein,Sample) -> all_sample_sep
all_sample_sep %>% merge(sample_metadata,by = "Sample") -> all_sample_sep_m_meta
all_sample_sep_m_meta %>% arrange(Enrichment,Eyzyme,Replicate) -> all_sample_sep_m_meta

Warning message:
“程辑包‘stringr’是用R版本4.3.2 来建造的”


In [2]:
merge_replicate <- function(df){
    # 合并重复
    df$Sample_merge_replicate <- gsub("21pcw(_1|_2|_3)", "21pcw", df$Sample)
    df %>% mutate(Sample=NULL) %>% rename(Sample=Sample_merge_replicate) -> df_1
    return(df_1)
}
df <- merge_replicate(all_sample_sep_m_meta)
protein_stats_v1 <- df %>%
  group_by(Protein) %>%
  summarise(
    Unique_Methods = n_distinct(Enrichment),  # 被多少种不同富集方法鉴定到
    Unique_Enzymes = n_distinct(Eyzyme),  # 被多少种不同酶切方法鉴定到
    Total_Observations = n(),                 # 被鉴定到的总次数
    Supported_Enrichments = paste(sort(unique(Enrichment)), collapse = ", "),  # 支持的富集方法（去重后逗号分隔）
    Supported_Enzymes = paste(sort(unique(Eyzyme)), collapse = ", ")  # 支持的酶切方法（去重后逗号分隔）
  )

In [4]:
head(protein_stats_v1)
nrow(protein_stats_v1)

Protein,Unique_Methods,Unique_Enzymes,Total_Observations,Supported_Enrichments,Supported_Enzymes
<chr>,<int>,<int>,<int>,<chr>,<chr>
ENST00000005340.10-chr17:7225351-7225558,1,1,1,C8,Trypsin_Chymotrypsin
ENST00000064778.8-chr11:73407530-73409496,1,1,1,PCP,Trypsin_LysC
ENST00000168216.11-chrX:53431403-53431813,1,1,1,PAGE,Trypsin_LysN
ENST00000199940.10+chr2:209730325-209730397,3,6,25,"C8, PAGE, PCP","Trypsin, Trypsin_ArgC, Trypsin_AspN, Trypsin_Chymotrypsin, Trypsin_LysC, Trypsin_LysN"
ENST00000215375.7+chr19:1241961-1244437,3,3,12,"C8, PAGE, PCP","Trypsin, Trypsin_LysC, Trypsin_LysN"
ENST00000216019.11-chr22:38485934-38485997,1,2,2,PCP,"Trypsin, Trypsin_ArgC"


[1] 2920

In [6]:
fwrite_c(protein_stats_v1[,"Protein"],"./stat_output/S7/protein.id.20250610.txt")

In [ ]:
system("bash S7.bio.sh")

In [13]:
read.table("./stat_output/S7/protein.id.20250610.tab") -> protein_id_seq
colnames(protein_id_seq) <- c("Protein","Seq")

In [17]:
merge(protein_stats_v1,protein_id_seq,by="Protein") -> protein_stats_v1_a_seq
nrow(protein_stats_v1_a_seq)

[1] 2920

In [21]:
protein_stats_v1_a_seq %>% filter(Unique_Methods==1) -> unique_methods_protein
unique_methods_protein[,c("Protein","Seq","Supported_Enrichments")] -> unique_methods_protein_df

In [22]:
table(unique_methods_protein_df$Supported_Enrichments)


  C8 MWCO PAGE  PCP 
 402  304  548  727 

In [25]:
summary(nchar(unique_methods_protein_df$Seq))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   6.00   34.00   66.00   71.39  108.00  150.00 

In [26]:
fwrite_c(unique_methods_protein_df,"./stat_output/S7/unique_methods_protein.txt")

## 只被一种酶切方法支持

In [34]:
protein_stats_v1_a_seq %>% filter(Unique_Enzymes==1) -> unique_enzymes_protein
nrow(unique_enzymes_protein)
head(unique_enzymes_protein)

[1] 1477

,Protein,Unique_Methods,Unique_Enzymes,Total_Observations,Supported_Enrichments,Supported_Enzymes,Seq
,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>
1,ENST00000005340.10-chr17:7225351-7225558,1,1,1,C8,Trypsin_Chymotrypsin,IGPLLTPECDLLLQLPTALPFAGLGVVSGTPGLGILDSSLRKLGDLGHVAAFVTLFILVTCIKHQIKQ
2,ENST00000064778.8-chr11:73407530-73409496,1,1,1,PCP,Trypsin_LysC,MSAGTLLTTPQHTAIGAHPVSMPTYRAQGTPAYSYVPPHW
3,ENST00000168216.11-chrX:53431403-53431813,1,1,1,PAGE,Trypsin_LysN,MTIAPGLFGTPLLTSLPEKVCNFLASQVPFPSRLGDPAEYAHLVQAIIENPFLNGEVIRLDGAIRMQP
4,ENST00000216019.11-chr22:38485934-38486048,1,1,1,PCP,Trypsin_GluC,MSQQFAQPPGATNMIGYMGQTAYQYPPPPPPPPPSRK
5,ENST00000216121.12-chr22:29555934-29561547,1,1,1,PAGE,Trypsin,MGPNIYELRTYKLKPGTMIEWGNNWARAIKYRQENQEAVGGFFSQIGELYVVHHLWAYKDLQSREETRNAAWRKRGWDENVYYTVPLVRHMESRIMIPLKISPLQ
6,ENST00000216129.7-chr22:43168007-43171851,1,1,3,PCP,Trypsin_Chymotrypsin,MNYDPDVVLKQVHCEEFIPEFEKQYPEFPWTDVQAEIFRAFTELFQVACAKPPPLGLCDYPSSRAMYAVDLMLKWDNGPDGRRVMQPQILEVNFNPDCERACRYHPTFFNDVFSTLFLDQPGGCHVTCLV


In [30]:
table(unique_enzymes_protein$Unique_Methods)
colnames(unique_enzymes_protein)


   1    2    3    4 
1339  103   30    5 

[1] "Protein"               "Unique_Methods"        "Unique_Enzymes"       
[4] "Total_Observations"    "Supported_Enrichments" "Supported_Enzymes"    
[7] "Seq"

In [35]:
unique_enzymes_protein[,c("Protein","Seq","Supported_Enzymes")] -> unique_enzymes_protein_df
fwrite_c(unique_enzymes_protein_df,"./stat_output/S7/unique_enzymes_protein.txt")

# 只被Trypsin_Chymotrypsin特异性鉴定到的小肽对应的肽段

In [38]:
# head(psm_sep_unique)
colnames(psm_sep_unique)

[1] "Spectrum"                    "Spectrum.File"              
 [3] "Peptide"                     "Modified.Peptide"           
 [5] "Extended.Peptide"            "Prev.AA"                    
 [7] "Next.AA"                     "Peptide.Length"             
 [9] "Charge"                      "Retention"                  
[11] "Observed.Mass"               "Calibrated.Observed.Mass"   
[13] "Observed.M.Z"                "Calibrated.Observed.M.Z"    
[15] "Calculated.Peptide.Mass"     "Calculated.M.Z"             
[17] "Delta.Mass"                  "SpectralSim"                
[19] "RTScore"                     "Expectation"                
[21] "Hyperscore"                  "Nextscore"                  
[23] "Probability"                 "Number.of.Enzymatic.Termini"
[25] "Number.of.Missed.Cleavages"  "Protein.Start"              
[27] "Protein.End"                 "Intensity"                  
[29] "Assigned.Modifications"      "Observed.Modifications"     
[31] "Class"                       "Ion.Mobility"               
[33] "Purity"                      "Is.Unique"                  
[35] "Protein"                     "Protein.ID"                 
[37] "Entry.Name"                  "Gene"                       
[39] "Protein.Description"         "Mapped.Genes"               
[41] "Mapped.Proteins"             "Sample"                     
[43] "Group_id"                    "PSM_type"

In [40]:
merge(psm_sep_unique,unique_enzymes_protein_df,by="Protein") -> unique_enzymes_protein_peptide

In [42]:
nrow(unique_enzymes_protein_peptide)

[1] 3589

In [44]:
fwrite(unique_enzymes_protein_peptide,"./stat_output//S7/unique_enzymes_protein_peptide.txt")